In [1]:
import numpy as np
import torch
from torch.autograd import Variable
import pennylane as qml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import expm
from scipy.stats import norm

In [2]:
# ═══════════════════════════════════════════════════════════════
# 1.  BLACK-SCHOLES HAMILTONIAN  (finite-difference, non-Hermitian)
# ═══════════════════════════════════════════════════════════════
 
def build_bs_hamiltonian(N: int, x_min: float, x_max: float,
                          sigma: float, r: float):
    """
    Build the non-Hermitian N×N tridiagonal H_BS in log-price basis.
 
    Central-difference scheme:
        ∂²/∂x² → (f_{i+1} - 2f_i + f_{i-1}) / dx²
        ∂/∂x   → (f_{i+1} - f_{i-1}) / (2 dx)
 
    Returns H (N×N) and x_grid (N,).
    """
    dx = (x_max - x_min) / (N - 1)
    x_grid = np.linspace(x_min, x_max, N)
    mu = sigma**2 / 2.0 - r        # risk-neutral drift
 
    a = sigma**2 / (2*dx**2) + mu / (2*dx)   # super-diagonal coeff
    b = sigma**2 / (2*dx**2) - mu / (2*dx)   # sub-diagonal coeff
    d = sigma**2 / dx**2 + r                  # diagonal
 
    H = (np.diag([d]*N)
       + np.diag([-a]*(N-1), k=+1)
       + np.diag([-b]*(N-1), k=-1))
    return H, x_grid

In [3]:
# ═══════════════════════════════════════════════════════════════
# 2.  SYMMETRISATION  (Baaquie §4.3 — similarity transform)
# ═══════════════════════════════════════════════════════════════
 
def symmetrise_hamiltonian(H: np.ndarray):
    """
    Given a tridiagonal H with super-diagonal a and sub-diagonal b (a≠b),
    find a diagonal D such that  H_sym = D⁻¹ H D  is symmetric.
 
    For tridiagonal matrices: D_i = (b/a)^{i/2}.
    This is the Baaquie similarity transform that removes the drift term
    from the Fokker-Planck / path-integral perspective.
 
    Returns (H_sym, D) where D is the diagonal (as a vector).
    """
    N = H.shape[0]
    # Infer a, b from off-diagonals
    a = -H[0, 1]    # super-diagonal value (should be uniform)
    b = -H[1, 0]    # sub-diagonal value
    ratio = np.sqrt(b / a)
    D = ratio**np.arange(N)
    H_sym = np.diag(1.0/D) @ H @ np.diag(D)
    return H_sym, D

In [4]:
# ═══════════════════════════════════════════════════════════════
# 3.  PAULI DECOMPOSITION  (PennyLane)
# ═══════════════════════════════════════════════════════════════
 
def pauli_decompose(H_sym: np.ndarray):
    """
    Express the Hermitian matrix H_sym as a weighted sum of Pauli strings:
        H_sym = Σ_k  c_k  P_k
    Returns (H_observable, coeffs, op_labels).
    """
    H_obs = qml.pauli_decompose(H_sym)
    coeffs_raw, ops_raw = H_obs.terms()
    coeffs = np.array([float(np.real(c)) for c in coeffs_raw])
    labels = [str(op) for op in ops_raw]
    return H_obs, coeffs, labels

In [5]:
# ═══════════════════════════════════════════════════════════════
# 4.  VQE — hardware-efficient ansatz, PyTorch back-end
#     (directly mirrors the Ising model PennyLane demo)
# ═══════════════════════════════════════════════════════════════
 
def build_vqe_qnode(n_qubits: int, n_layers: int):
    dev = qml.device("default.qubit", wires=n_qubits)
 
    @qml.qnode(dev, interface="torch")
    def circuit(params, observable):
        # Hardware-efficient ansatz: Ry rotations + CNOT chain
        for layer in range(n_layers):
            for q in range(n_qubits):
                qml.RY(params[layer, q], wires=q)
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])
        return qml.expval(observable)
 
    return circuit

In [6]:
def run_vqe(H_obs, n_qubits: int, n_layers: int = 3,
            n_steps: int = 300, lr: float = 0.05, seed: int = 42):
    """
    VQE using PyTorch SGD/Adam, exactly as in the PennyLane Ising demo.
    Returns (optimised_params_np, energy_history).
    """
    torch.manual_seed(seed)
    circuit = build_vqe_qnode(n_qubits, n_layers)
 
    params = Variable(
        (torch.rand(n_layers, n_qubits, dtype=torch.float64) * 2 * np.pi),
        requires_grad=True
    )
    opt = torch.optim.Adam([params], lr=lr)
    energies = []
 
    def closure():
        opt.zero_grad()
        loss = circuit(params, H_obs)
        loss.backward()
        return loss
 
    for step in range(n_steps):
        e = opt.step(closure)
        energies.append(float(e.detach()))
        if (step + 1) % 100 == 0:
            print(f"    step {step+1:4d}  |  E = {energies[-1]:.6f}")
 
    return params.detach().numpy(), energies

In [7]:
# ═══════════════════════════════════════════════════════════════
# 5.  OPTION PRICE FROM MATRIX-EXPONENTIAL PROPAGATOR
# ═══════════════════════════════════════════════════════════════
 
def call_payoff(x_grid: np.ndarray, K: float) -> np.ndarray:
    return np.maximum(np.exp(x_grid) - K, 0.0)
 
 
def price_via_propagator(H: np.ndarray, x_grid: np.ndarray,
                          K: float, T: float, S0: float) -> float:
    """
    Compute option price by directly exponentiating H_BS (classical, exact).
    V(x₀, 0) = e^{-r T} · ⟨x₀| e^{-T H_BS} |payoff⟩
    """
    payoff = call_payoff(x_grid, K)
    prop   = expm(-T * H)            # matrix exponential
    v_0   = prop @ payoff
    idx0  = np.argmin(np.abs(x_grid - np.log(S0)))
    return float(v_0[idx0])
 
 
def bs_price_exact(S0, K, r, sigma, T) -> float:
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

In [8]:
def plot_all(H_bs, H_sym, x_grid, eigvals, coeffs, labels,
             energies, vqe_E, price_exact, price_prop,
             Ns, prices_N, sigma, r, K, T, S0):
 
    DARK   = '#0d1117'
    PANEL  = '#161b27'
    GOLD   = '#f0c040'
    TEAL   = '#38d4c8'
    CORAL  = '#e06060'
    BLUE   = '#5590ee'
    MUTED  = '#7a8ba0'
    WHITE  = '#d8e4ee'
 
    fig = plt.figure(figsize=(15, 10), facecolor=DARK)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.40,
                            left=0.07, right=0.97, top=0.92, bottom=0.08)
 
    def style(ax, title):
        ax.set_facecolor(PANEL)
        ax.set_title(title, color=GOLD, fontsize=9.5, pad=7, fontweight='bold')
        ax.tick_params(colors=MUTED, labelsize=8)
        for sp in ax.spines.values():
            sp.set_edgecolor('#2a3545')
 
    # ── 1: H_BS heatmap ──
    ax = fig.add_subplot(gs[0, 0])
    im = ax.imshow(H_bs, cmap='RdYlBu_r', aspect='auto')
    cb = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.ax.tick_params(colors=MUTED, labelsize=7)
    labs = [f"{x:.2f}" for x in x_grid]
    ax.set_xticks(range(len(x_grid))); ax.set_xticklabels(labs, rotation=45, fontsize=7, color=MUTED)
    ax.set_yticks(range(len(x_grid))); ax.set_yticklabels(labs, fontsize=7, color=MUTED)
    ax.set_xlabel("log-price x", color=MUTED, fontsize=8)
    style(ax, "H_BS  (non-Hermitian, log-price basis)")
 
    # ── 2: eigenstates of H_sym ──
    ax2 = fig.add_subplot(gs[0, 1])
    S_vals = np.exp(x_grid)
    payoff = call_payoff(x_grid, K) / np.max(call_payoff(x_grid, K))
    ax2.fill_between(S_vals, payoff, alpha=0.15, color=GOLD)
    ax2.plot(S_vals, payoff, 'o-', color=GOLD, lw=2, ms=6, label='Payoff (norm.)')
    evecs = np.linalg.eigh(H_sym)[1]
    cols  = [TEAL, CORAL, BLUE]
    for i in range(3):
        v = evecs[:, i]; v /= np.max(np.abs(v))
        ax2.plot(S_vals, v, 'o--', color=cols[i], lw=1.5, alpha=0.85, ms=5,
                 label=f'ψ_{i}  E={eigvals[i]:.3f}')
    ax2.axvline(K, color=MUTED, lw=0.8, ls=':', alpha=0.6, label='Strike K')
    ax2.set_xlabel("Stock price S", color=MUTED, fontsize=8)
    ax2.set_ylabel("Amplitude", color=MUTED, fontsize=8)
    ax2.legend(fontsize=7, framealpha=0.25, labelcolor=WHITE,
               facecolor=PANEL, edgecolor='none', loc='upper left')
    style(ax2, "H_sym eigenstates  (after symmetrisation)")
 
    # ── 3: Pauli decomposition ──
    ax3 = fig.add_subplot(gs[0, 2])
    bar_c = [TEAL if c > 0 else CORAL for c in coeffs]
    ax3.bar(range(len(coeffs)), coeffs, color=bar_c, alpha=0.85, width=0.65, zorder=3)
    ax3.set_xticks(range(len(coeffs)))
    ax3.set_xticklabels(labels, rotation=40, ha='right', fontsize=8, color=MUTED)
    ax3.axhline(0, color=MUTED, lw=0.6, alpha=0.5)
    ax3.set_ylabel("Coefficient", color=MUTED, fontsize=8)
    ax3.grid(axis='y', color='#2a3545', lw=0.5, zorder=0)
    style(ax3, "Pauli decomposition of H_sym")
 
    # ── 4: VQE convergence ──
    ax4 = fig.add_subplot(gs[1, 0])
    steps = np.arange(len(energies))
    ax4.plot(steps, energies, color=TEAL, lw=1.8, label='VQE energy')
    ax4.axhline(eigvals[0], color=GOLD, lw=1.5, ls='--',
                label=f'Exact E₀ = {eigvals[0]:.4f}')
    ax4.fill_between(steps, energies, eigvals[0],
                     where=np.array(energies) > eigvals[0],
                     alpha=0.1, color=CORAL)
    ax4.set_xlabel("Optimisation step", color=MUTED, fontsize=8)
    ax4.set_ylabel("Energy ⟨H⟩", color=MUTED, fontsize=8)
    ax4.legend(fontsize=7.5, framealpha=0.25, labelcolor=WHITE,
               facecolor=PANEL, edgecolor='none')
    style(ax4, "VQE convergence  (Adam, PyTorch autograd)")
 
    # ── 5: Price convergence with N ──
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.plot(Ns, prices_N, 'o-', color=TEAL, lw=2, ms=7, zorder=3, label='Grid propagator')
    ax5.axhline(price_exact, color=GOLD, lw=1.5, ls='--',
                label=f'BS exact = {price_exact:.4f}')
    for nx, px in zip(Ns, prices_N):
        ax5.annotate(f'{px:.3f}', (nx, px), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=7, color=WHITE)
    ax5.set_xlabel("Grid points N", color=MUTED, fontsize=8)
    ax5.set_ylabel("Call price", color=MUTED, fontsize=8)
    ax5.set_xscale('log', base=2)
    ax5.legend(fontsize=7.5, framealpha=0.25, labelcolor=WHITE,
               facecolor=PANEL, edgecolor='none')
    ax5.grid(color='#2a3545', lw=0.5)
    style(ax5, "Price convergence with grid size N")
 
    # ── 6: BS landscape ──
    ax6 = fig.add_subplot(gs[1, 2])
    sv = np.linspace(0.05, 0.55, 80)
    Tv = np.linspace(0.1, 2.0, 80)
    SV, TV = np.meshgrid(sv, Tv)
    from scipy.stats import norm
    d1 = (np.log(S0/K) + (r + 0.5*SV**2)*TV)/(SV*np.sqrt(TV))
    prices_grid = S0*norm.cdf(d1) - K*np.exp(-r*TV)*norm.cdf(d1-SV*np.sqrt(TV))
    cf = ax6.contourf(sv*100, Tv, prices_grid, levels=12, cmap='plasma', alpha=0.85)
    cb2 = plt.colorbar(cf, ax=ax6, fraction=0.045, pad=0.03)
    cb2.ax.tick_params(colors=MUTED, labelsize=7)
    cb2.set_label('Call price', color=MUTED, fontsize=7)
    ax6.plot(sigma*100, T, '*', color='white', ms=12, zorder=5, label=f'Our params')
    ax6.set_xlabel("Volatility σ (%)", color=MUTED, fontsize=8)
    ax6.set_ylabel("Time to maturity T", color=MUTED, fontsize=8)
    ax6.legend(fontsize=7.5, framealpha=0.25, labelcolor=WHITE,
               facecolor=PANEL, edgecolor='none')
    style(ax6, "BS call price  (σ × T landscape)")
 
    fig.suptitle(
        "European Call Option · Baaquie Quantum Hamiltonian · PennyLane VQE  "
        f"(S₀={S0}, K={K}, σ={sigma}, r={r}, T={T})",
        color=GOLD, fontsize=12, fontweight='bold'
    )
 
    plt.savefig("images/quantum_option_result.png",
                dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.close()

In [9]:
# ═══════════════════════════════════════════════════════════════
# 6.  FULL PIPELINE
# ═══════════════════════════════════════════════════════════════
 
def main():
    print("=" * 62)
    print("  Quantum European Call Option — Baaquie Hamiltonian + VQE")
    print("=" * 62)
 
    S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.20, 1.0
    n_qubits = 5          # 4 grid points
    N = 2**n_qubits
    x_min = np.log(S0) - 3*sigma*np.sqrt(T)
    x_max = np.log(S0) + 3*sigma*np.sqrt(T)
 
    print(f"\n  S₀={S0}  K={K}  r={r}  σ={sigma}  T={T}")
    print(f"  Grid: N={N} points  |  Qubits: {n_qubits}")
 
    # Step 1 — H_BS
    print("\n[1] Building Black-Scholes Hamiltonian ...")
    H_bs, x_grid = build_bs_hamiltonian(N, x_min, x_max, sigma, r)
    print(f"    H_BS eigenvalues (real part): "
          f"{np.round(np.linalg.eigvals(H_bs).real, 4)}")
 
    # Step 2 — Symmetrize
    print("\n[2] Symmetrising via Baaquie similarity transform ...")
    H_sym, D_vec = symmetrise_hamiltonian(H_bs)
    eigvals = np.linalg.eigvalsh(H_sym)
    print(f"    Symmetry residual: {np.max(np.abs(H_sym - H_sym.T)):.2e}")
    print(f"    H_sym eigenvalues: {np.round(eigvals, 4)}")
    print(f"    Ground energy E₀ (classical): {eigvals[0]:.6f}")
 
    # Step 3 — Pauli decomposition
    print("\n[3] Pauli decomposition ...")
    H_obs, coeffs, labels = pauli_decompose(H_sym)
    print(f"    {len(coeffs)} Pauli terms:")
    for c, l in zip(coeffs, labels):
        print(f"      {c:+.5f}  ×  {l}")
 
    # Step 4 — VQE
    print("\n[4] VQE (Adam / PyTorch, mirroring PennyLane Ising demo) ...")
    opt_params, energies = run_vqe(H_obs, n_qubits, n_layers=3,
                                    n_steps=400, lr=0.04)
    vqe_E = energies[-1]
    print(f"\n    VQE ground energy:      {vqe_E:.6f}")
    print(f"    Classical ground energy: {eigvals[0]:.6f}")
    print(f"    Absolute error:          {abs(vqe_E - eigvals[0]):.2e}")
 
    # Step 5 — Pricing
    print("\n[5] Option pricing ...")
    price_prop  = price_via_propagator(H_bs, x_grid, K, T, S0)
    price_exact = bs_price_exact(S0, K, r, sigma, T)
    print(f"    Black-Scholes exact:           {price_exact:.4f}")
    print(f"    Matrix-exponential propagator: {price_prop:.4f}")
    print(f"    Coarse-grid error:             {abs(price_prop - price_exact):.4f}")
    print("    (Error shrinks rapidly with N; use N=64 or 128 for production)")
 
    # Convergence study
    Ns = [2**x for x in range(1,n_qubits+1)]
    prices_N = []
    for Ni in Ns:
        Hi, xi = build_bs_hamiltonian(Ni, x_min - sigma, x_max + sigma, sigma, r)
        pi = price_via_propagator(Hi, xi, K, T, S0)
        prices_N.append(pi)
        print(f"    N={Ni:3d}  price={pi:.4f}  err={abs(pi-price_exact):.4f}")
 
    # Plots
    print("\n[6] Plotting ...")
    plot_all(H_bs, H_sym, x_grid, eigvals, coeffs, labels,
             energies, vqe_E, price_exact, price_prop,
             Ns, prices_N, sigma, r, K, T, S0)
    print("\nSaved → quantum_option_result.png")
    return energies, vqe_E, eigvals

In [10]:
if __name__ == "__main__":
    main()

  Quantum European Call Option — Baaquie Hamiltonian + VQE

  S₀=100.0  K=100.0  r=0.05  σ=0.2  T=1.0
  Grid: N=32 points  |  Qubits: 5

[1] Building Black-Scholes Hamiltonian ...
    H_BS eigenvalues (real part): [53.3068 52.9454 52.3468 51.5163 50.4614 49.1918 47.7189 46.056  44.2182
 42.2222 40.086  37.829  35.4717 33.0352 30.5419 28.0141 25.4748 22.947
 20.4536 18.0172 15.6598  0.1821  0.5435  1.1421  1.9726  3.0275  4.2971
  5.77    7.4329  9.2707 11.2667 13.4028]

[2] Symmetrising via Baaquie similarity transform ...
    Symmetry residual: 8.88e-15
    H_sym eigenvalues: [ 0.1821  0.5435  1.1421  1.9726  3.0275  4.2971  5.77    7.4329  9.2707
 11.2667 13.4028 15.6598 18.0172 20.4536 22.947  25.4748 28.0141 30.5419
 33.0352 35.4717 37.829  40.086  42.2222 44.2182 46.056  47.7189 49.1918
 50.4614 51.5163 52.3468 52.9454 53.3068]
    Ground energy E₀ (classical): 0.182076

[3] Pauli decomposition ...
    32 Pauli terms:
      +26.74444  ×  I(0) @ I(1) @ I(2) @ I(3) @ I(4)
      -13.